 ## Try labels from different experts

## Load and process raw data

In [ ]:
import pandas as pd
import numpy as np

base_dir = "../data/processed/Y/CT_POST"

ROI_path_JM1 = f"{base_dir}/ROI post op ct JMG.xlsx"

ROI_path_Dan1 = f"{base_dir}/ROI postop Dan Youssef.xlsx"

ROI_path_Bridget1 = f"{base_dir}/Post op ROI Bridget.xlsx"

roi_CT_JM1 = pd.read_excel(ROI_path_JM1, index_col=[0, 1, 2], header=None)

roi_CT_Dan1 = pd.read_excel(ROI_path_Dan1, index_col=[0, 1, 2], header=None)

roi_CT_Bridget1 = pd.read_excel(ROI_path_Bridget1, index_col=[0, 1, 2], header=None)

In [ ]:
index = roi_CT_Bridget1.index

In [ ]:
ck_pt = 'SM'
ck_l = 'LLSCC post'

print(roi_CT_JM1.loc[(ck_pt, 'CT',  ck_l)].values.reshape(1,3))
print(roi_CT_Dan1.loc[(ck_pt, 'CT',  ck_l)].values.reshape(1,3))
print(roi_CT_Bridget1.loc[(ck_pt, 'CT',  ck_l)].values.reshape(1,3))

In [ ]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))

if module_path not in sys.path:
    sys.path.append(module_path)

sys.path

In [ ]:
import src.models.common.MyDataset as MyDataset

names = MyDataset.get_pat_names()
names.remove("DM2")
names.remove("HH")
names.remove("JM")

landmarks = ['LLSCC ant', 'LLSCC post', 'RLSCC ant', 'RLSCC post']

all_labels = []

for pt_name in names:
    for landmark in landmarks:
        l1 = roi_CT_JM1.loc[(pt_name, 'CT',  landmark)].values.reshape(1,3)
        l3 = roi_CT_Dan1.loc[(pt_name, 'CT',  landmark)].values.reshape(1,3)
        l5 = roi_CT_Bridget1.loc[(pt_name, 'CT',  landmark)].values.reshape(1,3)
        one_landmark = np.concatenate([l1, l3, l5], axis=0)
        if len(all_labels) == 0:
            all_labels = one_landmark
        else:
            all_labels = np.concatenate([all_labels, one_landmark], axis=0)

In [ ]:
all_labels = all_labels.reshape((17, 4, 3, 3))
print(all_labels[13, 1, :, :])
print(all_labels[0:4])

In [ ]:
medium_labels = np.median(all_labels, axis=2)

In [ ]:
medium_labels_tmp = np.reshape(medium_labels, (68, 3))
print(medium_labels[0:4])

In [ ]:
all_labels_reshape = all_labels.reshape((68, 9))
save_columns = ['x', 'y', 'z', 'x', 'y', 'z', 'x', 'y', 'z']
save_index = np.asarray(['LLSCC ant', 'LLSCC post', 'RLSCC ant', 'RLSCC post'])
save_index = np.repeat(save_index.reshape(1, 4), 17, axis=0).reshape(68)

In [ ]:
 # save the processed Y
df1 = pd.DataFrame(all_labels_reshape,
                   index=save_index,
                   columns=save_columns)
df1.to_excel(f"{base_dir}/ROI_3.xlsx")

In [ ]:
np.save(f"{base_dir}/ROI_CT_Post_Medium", medium_labels)

In [ ]:
save_columns = ['x', 'y', 'z']
save_index = np.asarray(['LLSCC ant', 'LLSCC post', 'RLSCC ant', 'RLSCC post'])
save_index = np.repeat(save_index.reshape(1, 4), 17, axis=0).reshape(68)

In [ ]:
 # save the processed Y
df2 = pd.DataFrame(medium_labels_tmp,
                   index=save_index,
                   columns=save_columns)
df2.to_excel(f"{base_dir}/ROI_medium.xlsx")

## Check the medium ROI

In [ ]:
import imageio as iio
import scipy.ndimage as ndi
import numpy as np
import matplotlib.pyplot as plt

import src.models.common.Visualization as Visualization
import src.models.common.MyDataset as MyDataset

names = MyDataset.get_pat_names()
names.remove("DM2")
names.remove("HH")
names.remove("JM")

pt_id = 16
pt_name = names[pt_id]

dicom_AH_MR_path = f"/data/gpfs/projects/punim1836/Data/raw/CT_MRI_Pre_Post/{pt_name} Post"
head_vol = iio.volread(dicom_AH_MR_path, 'DICOM')

head_vol.shape

sampling = head_vol.meta['sampling']
pixel_space = [sampling[1], sampling[2], sampling[0]]
print(pixel_space)

head_vol_array = np.moveaxis(np.asarray(head_vol), 0 ,2)

print("roi: ", medium_labels[pt_id])
Visualization.show_pts(head_vol_array, medium_labels[pt_id], pixel_space)

## Check the experts' div (not yet for CT Post)

In [ ]:
medium_labels = medium_labels.reshape((20, 4, 1, 3))
medium_labels_rep = np.repeat(medium_labels, 6, axis=2)

In [ ]:
diff_all_medium = (all_labels - medium_labels_rep) * 0.15

diff_p2 = np.power(diff_all_medium, 2)

diff_dis = np.power(np.sum(diff_p2, axis=3), 1/2)

print(np.mean(diff_dis))
print(np.std(diff_dis))

In [ ]:
# base_save_dir = "../data/processed/Y"
# 
# np.save(f"{base_save_dir}/ROI_CT_Medium", medium_labels)

In [ ]:
# Anterior and Posterior differences from the labels
ant_diff_dis = diff_dis[:, [0, 2], :]
pos_diff_dis = diff_dis[:, [1, 3], :]

ant_diff_dis_mean = np.mean(ant_diff_dis)
ant_diff_dis_dev = np.std(ant_diff_dis)
pos_diff_dis_mean = np.mean(pos_diff_dis)
pos_diff_dis_dev = np.std(pos_diff_dis)

print(f"Anterior mean: [{ant_diff_dis_mean}], dev: [{ant_diff_dis_dev}]")
print(f"Posterior mean: [{pos_diff_dis_mean}], dev: [{pos_diff_dis_dev}]")

## try to upgrade from ct_medium_3 to ct_medium_6 ...

In [ ]:
import numpy as np

ct_medium_6 = np.load("../data/processed/Y/ROI_CT_Medium.npy")
ct_medium_3 = np.load("../data/raw/Y/ROI_3/ROI_CT_Medium.npy")

diff = ct_medium_3 - ct_medium_6

In [ ]:
#diff[:, 2:4, 0] = - diff[:, 2:4, 0]
print(diff[0:5])

In [ ]:
diff_flip = np.copy(diff)
diff_flip[:, 2:4, 0] = -diff_flip[:, 2:4, 0]
print(diff_flip[0:5])

In [ ]:
np.save("../Resources/diff_flip", diff_flip)

In [ ]:
import numpy as np
import importlib
import Functions.MyDataset as MyDataset
import Functions.Visualization as Visualization

pat_aug_path = "/Volumes/Shawn_HDD/PhD/Project/Date/augmentation_from_matlab/original_augmentation_data/JM_aug_1.mat"

pixel_space = [0.15, 0.15, 0.15]

importlib.reload(Visualization)

pat_aug_volume, pat_aug_pts, _ = MyDataset.load_mat_data(pat_aug_path)

print(pat_aug_pts)

Visualization.show_pts(pat_aug_volume, pat_aug_pts, pixel_space)

print(pat_aug_pts)

In [ ]:
Visualization.show_pts(pat_aug_volume, medium_labels[13, :, :], pixel_space)